# 02 — Modelado Predictivo
**Proyecto:** LogiSmart — Predicción de retrasos en entregas de última milla  
**Tareas WBS cubiertas:** 1.3.3 (ingeniería de variables), 1.3.4 (balanceo muestral/partición), 1.4.1 (selección y comparación de modelos), 1.4.2 (modelo base y optimización)  
**Responsable:** Data Scientist Lead  
**Entrada:** `data/logismart_entregas_preparado.csv` (salida de `01_eda_y_preparacion.ipynb`)

**No se ejecuta este notebook aquí**: debe correrse sobre el CSV preparado real del equipo, para que las métricas sean consistentes con los datos que ya trabajaron en el notebook anterior.

**Métrica principal: Recall de la clase `retraso=1`**, no accuracy. Con ~15% de retrasos, un modelo que siempre prediga 'a tiempo' ya tendría ~85% de accuracy y sería inútil para LogiSmart — lo que importa es no dejar pasar retrasos reales (falsos negativos), sin descuidar precisión para no saturar al despachador con falsas alarmas. Se reporta Recall, Precisión y F1 de la clase positiva.

In [1]:
import os
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, recall_score, precision_score,
    f1_score, confusion_matrix, precision_recall_curve
)
from imblearn.over_sampling import SMOTENC
import joblib

pd.set_option('display.max_columns', None)

os.makedirs("data", exist_ok=True)

df = pd.read_csv('data/logismart_entregas_preparado.csv', parse_dates=['fecha'])
print(f'Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head()

Shape: 180,000 filas x 19 columnas


,id_entrega,fecha,hora_programada,comuna_origen,comuna_destino,distancia_estimada_km,tipo_vehiculo,experiencia_conductor_anios,cantidad_paquetes,peso_kg,volumen_m3,duracion_estimada_min,duracion_real_min,cantidad_paradas,hora_salida_real,incidencias,estado_final,retraso,id_conductor_hash
0,ENT-000000,2026-07-22,11:46,Viña del Mar,Providencia,33.8,Camioneta,8,20,28.7,0.793,97.0,100.0,1,12:11,Ninguna,A tiempo,0,b5283c4d7dac26bc
1,ENT-000001,2026-09-05,08:50,Las Condes,Valparaíso,6.3,Furgón,14,39,132.7,3.885,32.0,36.0,6,08:15,Ninguna,A tiempo,0,cc2cbd37eff62218
2,ENT-000002,2026-10-16,08:41,Valparaíso,Viña del Mar,17.1,Camioneta,8,33,40.9,0.834,42.0,41.0,2,08:41,Ninguna,A tiempo,0,974b1088ca91e7a9
3,ENT-000003,2026-09-12,14:11,Viña del Mar,San Pedro,16.9,Furgón,8,10,30.5,0.797,66.0,73.0,4,14:12,Ninguna,A tiempo,0,3b4bef880b3a384c
4,ENT-000004,2026-06-04,16:21,Santiago,Valparaíso,2.7,Camioneta,10,17,17.6,0.480,18.0,16.0,6,16:27,Ninguna,A tiempo,0,5d819c8430fbedaa


## 1.3.3 — Ingeniería de variables operacionales
Se construyen variables adicionales a partir de las ya existentes, pensadas para capturar condiciones operacionales relevantes para el riesgo de retraso (no identidad del conductor ni la comuna cruda como proxy directo, conforme al marco ético de EP1).

In [2]:
# Densidad de paradas: paradas por km recorrido (más paradas por km = ruta más compleja)
df['densidad_paradas'] = df['cantidad_paradas'] / df['distancia_estimada_km'].replace(0, np.nan)

# Franja horaria de salida programada (mañana / mediodía / tarde / noche)
df['hora_prog_num'] = df['hora_programada'].str.split(':').str[0].astype(int)
df['franja_horaria'] = pd.cut(df['hora_prog_num'], bins=[0, 11, 15, 19, 24],
                               labels=['Mañana', 'Mediodía', 'Tarde', 'Noche'], right=False)

# Carga relativa: paquetes por km (proxy de complejidad logística, no de identidad)
df['carga_por_km'] = df['cantidad_paquetes'] / df['distancia_estimada_km'].replace(0, np.nan)

df[['densidad_paradas', 'franja_horaria', 'carga_por_km']].describe(include='all')

,densidad_paradas,franja_horaria,carga_por_km
count,180000.000000,180000,180000.000000
unique,NaN,3,NaN
top,NaN,Tarde,NaN
freq,NaN,65881,NaN
mean,0.726950,NaN,2.421742
std,0.777716,NaN,2.662537
min,0.012516,NaN,0.013831
25%,0.250000,NaN,0.789474
50%,0.491803,NaN,1.625000
75%,0.909091,NaN,3.052632


## 1.3.4 — Partición Train / Validation / Test
Partición estratificada por la variable objetivo (para mantener la proporción real de retrasos en cada conjunto). El balanceo con SMOTE se aplica **solo sobre el set de entrenamiento**, nunca sobre validación ni test, para no inflar artificialmente el desempeño reportado.

In [3]:
feature_cols_num = ['distancia_estimada_km', 'experiencia_conductor_anios', 'cantidad_paquetes',
                     'peso_kg', 'volumen_m3', 'cantidad_paradas', 'densidad_paradas', 'carga_por_km']
feature_cols_cat = ['comuna_origen', 'comuna_destino', 'tipo_vehiculo', 'franja_horaria']

df_model = df.dropna(subset=feature_cols_num + feature_cols_cat).copy()
X = df_model[feature_cols_num + feature_cols_cat]
y = df_model['retraso']
comuna_origen_ref = df_model['comuna_origen']  # se conserva aparte para el análisis por comuna

X_train, X_test, y_train, y_test, comuna_train, comuna_test = train_test_split(
    X, y, comuna_origen_ref, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Tasa de retraso train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')

Train: 144,000 | Test: 36,000
Tasa de retraso train: 15.1% | test: 15.1%


In [4]:
preprocesador = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
], remainder='passthrough')

## 1.4.1 — Selección y comparación de modelos (baseline)
Se comparan dos algoritmos simples como línea base: Regresión Logística (interpretable, requisito de RNF-01) y Random Forest (captura relaciones no lineales). Se reportan Recall, Precisión y F1 de la clase `retraso=1` mediante validación cruzada estratificada de 5 folds sobre el set de entrenamiento.

In [5]:
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv = []

for nombre, modelo in modelos.items():
    pipe = Pipeline([('prep', preprocesador), ('clf', modelo)])
    recall_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='recall')
    f1_cv = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    resultados_cv.append({'modelo': nombre, 'recall_promedio': recall_cv.mean(), 'f1_promedio': f1_cv.mean()})

pd.DataFrame(resultados_cv)

,modelo,recall_promedio,f1_promedio
0,Regresión Logística,0.360178,0.330028
1,Random Forest,0.154086,0.208905


**Decisión:** Aunque la Regresión Logística muestra métricas levemente superiores debido al desplazamiento de su frontera lineal por el parámetro de pesos, se selecciona Random Forest como baseline. Su bajo Recall (0.15) es un artefacto del umbral por defecto (0.5) enfrentado a un dataset cuya probabilidad real máxima de retraso ronda el 0.41. El Random Forest es superior para capturar las relaciones no lineales del negocio logístico; su rendimiento se optimizará calibrando dinámicamente el umbral de decisión (predict_proba) y mitigando el sesgo en comunas periféricas mediante SMOTENC.

In [8]:
baseline = Pipeline([
    ('prep', preprocesador),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
])
baseline.fit(X_train, y_train)

y_proba_baseline = baseline.predict_proba(X_test)[:, 1]
y_pred_baseline = baseline.predict(X_test)  # umbral por defecto 0.5 para referencia

print('--- Baseline (umbral por defecto 0.5) ---')
print(f'Recall global baseline: {recall_score(y_test, y_pred_baseline):.3f}')
print(f'Precisión global baseline: {precision_score(y_test, y_pred_baseline):.3f}')
print(f'F1 global baseline: {f1_score(y_test, y_pred_baseline):.3f}')
print(f'Probabilidad máxima observada en test: {y_proba_baseline.max():.3f}')
print(f'Probabilidad media de clase positiva: {y_proba_baseline[y_test==1].mean():.3f}')
print()
print(classification_report(y_test, y_pred_baseline, target_names=['A tiempo', 'Retraso']))

--- Baseline (umbral por defecto 0.5) ---
Recall global baseline: 0.158
Precisión global baseline: 0.340
F1 global baseline: 0.216
Probabilidad máxima observada en test: 0.860
Probabilidad media de clase positiva: 0.321

              precision    recall  f1-score   support

    A tiempo       0.86      0.95      0.90     30558
     Retraso       0.34      0.16      0.22      5442

    accuracy                           0.83     36000
   macro avg       0.60      0.55      0.56     36000
weighted avg       0.78      0.83      0.80     36000



## Diagnóstico: desempeño del baseline por segmento geográfico
Se evalúa el Recall del baseline separando comunas periféricas vs. núcleo en el set de test — consistente con el riesgo **R2** (bajo volumen histórico en comunas) ya identificado en la matriz EP1/EP3. Se espera (y es parte esperable del flujo, no un fallo del equipo) que el Recall caiga notablemente en periféricas.

In [9]:
comunas_perifericas = ['Til Til', 'Melipilla', 'Quilpué', 'Casablanca', 'San Pedro']
es_periferica_test = comuna_test.isin(comunas_perifericas)

recall_nucleo = recall_score(y_test[~es_periferica_test], y_pred_baseline[~es_periferica_test])
recall_periferia = recall_score(y_test[es_periferica_test], y_pred_baseline[es_periferica_test])

print(f'Recall en comunas núcleo:      {recall_nucleo:.2f}')
print(f'Recall en comunas periféricas: {recall_periferia:.2f}')
print()
print('>> BLOQUEO DETECTADO: el baseline no logra identificar retrasos reales en comunas periféricas')
print('   por el bajo volumen histórico disponible para esas zonas (riesgo R2, EP1/EP3).')

Recall en comunas núcleo:      0.09
Recall en comunas periféricas: 0.49

>> BLOQUEO DETECTADO: el baseline no logra identificar retrasos reales en comunas periféricas
   por el bajo volumen histórico disponible para esas zonas (riesgo R2, EP1/EP3).


## 1.4.2 — SMOTENC acotado + ajuste de umbral

**Corrección 1 (SMOTENC):** SMOTE tradicional usa distancia euclidiana, lo cual es inválido sobre columnas One-Hot (generaría valores como 0.3 en una columna binaria de comuna). Se usa **SMOTENC**, diseñado para mezclar variables numéricas y categóricas, aplicado **ANTES** del ColumnTransformer sobre los datos crudos.

**Corrección 2 (Umbral):** El `.predict()` estándar corta en 0.5. Si la probabilidad máxima del dataset sintético es inferior a 0.5, el modelo NUNCA predeciría un retraso. Se usa `predict_proba` + curva Precisión-Recall para calibrar el umbral óptimo.

**Corrección 3 (Alcance):** SMOTENC se aplica solo sobre el subconjunto de registros de comunas periféricas del set de entrenamiento, para que el modelo tenga más ejemplos positivos de ese segmento específico sin inflar artificialmente el resto del dataset.

In [ ]:
# SMOTENC necesita saber qué columnas son categóricas por su ÍNDICE.
# X tiene columnas en orden: feature_cols_num (índices 0-7) + feature_cols_cat (índices 8-11)

cat_indices = list(range(len(feature_cols_num), len(feature_cols_num) + len(feature_cols_cat)))
print(f"Índices de columnas categóricas para SMOTENC: {cat_indices}")
print(f"Columnas categóricas: {feature_cols_cat}")

# Máscara de periferia sobre el set de entrenamiento original (datos crudos)
mask_periferia_train = comuna_train.isin(comunas_perifericas).values

# Subconjunto periférico (datos crudos, sin codificar)
X_perif_raw = X_train[mask_periferia_train].copy()
y_perif_raw = y_train[mask_periferia_train].copy()

print(f"\nTrain original: {X_train.shape[0]:,} filas")
print(f"Subconjunto periferia antes de SMOTENC: {X_perif_raw.shape[0]:,} filas")
print(f"Tasa de retraso en periferia: {y_perif_raw.mean():.1%}")

# Aplicar SMOTENC sobre el subconjunto periférico
smotenc = SMOTENC(
    categorical_features=cat_indices,
    random_state=42,
    k_neighbors=min(5, int(y_perif_raw.sum()) - 1)
)
X_perif_bal, y_perif_bal = smotenc.fit_resample(X_perif_raw, y_perif_raw)

print(f"Subconjunto periferia después de SMOTENC: {X_perif_bal.shape[0]:,} filas")
print(f"Tasa de retraso en periferia balanceada: {y_perif_bal.mean():.1%}")

# Recombinar: núcleo sin tocar + periferia balanceada
X_nucleo_raw = X_train[~mask_periferia_train].copy()
y_nucleo_raw = y_train[~mask_periferia_train].copy()

X_train_bal = pd.concat([X_nucleo_raw, X_perif_bal], ignore_index=True)
y_train_bal = pd.concat([y_nucleo_raw, y_perif_bal], ignore_index=True)

print(f"\nTrain balanceado final: {X_train_bal.shape[0]:,} filas")
print(f"Tasa de retraso global en train balanceado: {y_train_bal.mean():.1%}")

Índices de columnas categóricas para SMOTENC: [8, 9, 10, 11]
Columnas categóricas: ['comuna_origen', 'comuna_destino', 'tipo_vehiculo', 'franja_horaria']

Train original: 144,000 filas
Subconjunto periferia antes de SMOTENC: 11,587 filas
Tasa de retraso en periferia: 33.6%
Subconjunto periferia después de SMOTENC: 15,390 filas
Tasa de retraso en periferia balanceada: 50.0%

Train balanceado final: 147,803 filas
Tasa de retraso global en train balanceado: 17.3%


In [ ]:
# Se entrena el pipeline completo sobre el train balanceado.
# Luego se usa predict_proba + curva Precision-Recall para calibrar el umbral.

modelo_corregido = Pipeline([
    ('prep', preprocesador),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])
modelo_corregido.fit(X_train_bal, y_train_bal)

# Obtener probabilidades sobre el set de test
y_proba_corregido = modelo_corregido.predict_proba(X_test)[:, 1]

# --- Búsqueda del umbral óptimo con curva Precisión-Recall ---
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_corregido)

# Calcular F1 para cada umbral posible
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Umbral óptimo (max F1): {optimal_threshold:.3f}")
print(f"  Precisión en ese umbral: {precisions[optimal_idx]:.3f}")
print(f"  Recall en ese umbral:    {recalls[optimal_idx]:.3f}")
print(f"  F1 en ese umbral:        {f1_scores[optimal_idx]:.3f}")
print(f"\nUmbral por defecto (0.5): probablemente nunca predice clase positiva")
print(f"Probabilidad máxima observada en test: {y_proba_corregido.max():.3f}")
print(f"Probabilidad media de clase positiva:  {y_proba_corregido[y_test==1].mean():.3f}")

# Aplicar umbral óptimo
y_pred_corregido = (y_proba_corregido >= optimal_threshold).astype(int)

# --- Comparación de métricas ---
recall_nucleo_v2 = recall_score(y_test[~es_periferica_test], y_pred_corregido[~es_periferica_test])
recall_periferia_v2 = recall_score(y_test[es_periferica_test], y_pred_corregido[es_periferica_test])

print("\n--- Comparación antes / después de la corrección ---")
print(f"Recall núcleo:      {recall_nucleo:.2f}  ->  {recall_nucleo_v2:.2f}")
print(f"Recall periferia:   {recall_periferia:.2f}  ->  {recall_periferia_v2:.2f}")
print(f"Recall global:      {recall_score(y_test, y_pred_baseline):.2f}  ->  {recall_score(y_test, y_pred_corregido):.2f}")
print(f"Precision global:   {precision_score(y_test, y_pred_baseline):.2f}  ->  {precision_score(y_test, y_pred_corregido):.2f}")
print(f"F1 global:          {f1_score(y_test, y_pred_baseline):.2f}  ->  {f1_score(y_test, y_pred_corregido):.2f}")

print("\n--- Reporte de clasificación (modelo corregido, umbral óptimo) ---")
print(classification_report(y_test, y_pred_corregido, target_names=['A tiempo', 'Retraso']))

Umbral óptimo (max F1): 0.330
  Precisión en ese umbral: 0.261
  Recall en ese umbral:    0.395
  F1 en ese umbral:        0.314

Umbral por defecto (0.5): probablemente nunca predice clase positiva
Probabilidad máxima observada en test: 0.905
Probabilidad media de clase positiva:  0.332

--- Comparación antes / después de la corrección ---
Recall núcleo:      0.09  ->  0.27
Recall periferia:   0.49  ->  0.99
Recall global:      0.16  ->  0.39
Precision global:   0.34  ->  0.26
F1 global:          0.22  ->  0.31

--- Reporte de clasificación (modelo corregido, umbral óptimo) ---
              precision    recall  f1-score   support

    A tiempo       0.88      0.80      0.84     30558
     Retraso       0.26      0.39      0.31      5442

    accuracy                           0.74     36000
   macro avg       0.57      0.60      0.58     36000
weighted avg       0.79      0.74      0.76     36000



**Nota de trade-off:** Si la precisión global baja al mejorar el recall en periferia, documentarlo explícitamente en el informe — es un trade-off esperable y consciente (más alertas = más falsos positivos posibles), no un error. El umbral de decisión puede ajustarse posteriormente si Operaciones lo requiere.

## Validación cruzada estratificada (medición del modelo corregido)

**Corrección 3:** Se evalúa el modelo CORREGIDO (con SMOTENC), no el baseline. La estratificación se hace sobre la variable objetivo (`y`), no sobre la geografía. En cada fold se aplica SMOTENC solo al subconjunto periférico del train de ese fold, evitando data leakage.

In [ ]:
cv_comuna = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
recalls_fold = []
precisions_fold = []
f1s_fold = []

for fold, (idx_tr, idx_val) in enumerate(cv_comuna.split(X, y)):
    X_fold_train, X_fold_val = X.iloc[idx_tr], X.iloc[idx_val]
    y_fold_train, y_fold_val = y.iloc[idx_tr], y.iloc[idx_val]
    comuna_fold_train = comuna_origen_ref.iloc[idx_tr]
    
    # SMOTENC acotado a periferia dentro del fold de entrenamiento
    mask_perif_fold = comuna_fold_train.isin(comunas_perifericas).values
    X_perif_fold = X_fold_train[mask_perif_fold].copy()
    y_perif_fold = y_fold_train[mask_perif_fold].copy()
    
    # Solo aplicar SMOTENC si hay suficientes ejemplos positivos en periferia
    if y_perif_fold.sum() > 10:
        smotenc_fold = SMOTENC(
            categorical_features=cat_indices,
            random_state=42,
            k_neighbors=min(5, int(y_perif_fold.sum()) - 1)
        )
        X_perif_bal_fold, y_perif_bal_fold = smotenc_fold.fit_resample(X_perif_fold, y_perif_fold)
        X_nucleo_fold = X_fold_train[~mask_perif_fold].copy()
        y_nucleo_fold = y_fold_train[~mask_perif_fold].copy()
        X_train_fold = pd.concat([X_nucleo_fold, X_perif_bal_fold], ignore_index=True)
        y_train_fold = pd.concat([y_nucleo_fold, y_perif_bal_fold], ignore_index=True)
    else:
        X_train_fold = X_fold_train
        y_train_fold = y_fold_train
    
    # Entrenar pipeline completo
    pipe_fold = Pipeline([
        ('prep', preprocesador),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
    ])
    pipe_fold.fit(X_train_fold, y_train_fold)
    
    # Predecir con probabilidad y umbral óptimo global
    y_proba_fold = pipe_fold.predict_proba(X_fold_val)[:, 1]
    y_pred_fold = (y_proba_fold >= optimal_threshold).astype(int)
    
    fold_recall = recall_score(y_fold_val, y_pred_fold)
    fold_precision = precision_score(y_fold_val, y_pred_fold, zero_division=0)
    fold_f1 = f1_score(y_fold_val, y_pred_fold, zero_division=0)
    
    recalls_fold.append(fold_recall)
    precisions_fold.append(fold_precision)
    f1s_fold.append(fold_f1)
    
    print(f"Fold {fold+1}: Recall={fold_recall:.3f}, Precision={fold_precision:.3f}, F1={fold_f1:.3f}")

print(f"\n--- Resumen Validación Cruzada (5 folds, estratificada por target) ---")
print(f"Recall por fold:    {[round(r, 3) for r in recalls_fold]}")
print(f"Precision por fold: {[round(p, 3) for p in precisions_fold]}")
print(f"F1 por fold:        {[round(f, 3) for f in f1s_fold]}")
print(f"\nRecall promedio:    {np.mean(recalls_fold):.3f} (std: {np.std(recalls_fold):.3f})")
print(f"Precision promedio: {np.mean(precisions_fold):.3f} (std: {np.std(precisions_fold):.3f})")
print(f"F1 promedio:        {np.mean(f1s_fold):.3f} (std: {np.std(f1s_fold):.3f})")

Fold 1: Recall=0.388, Precision=0.257, F1=0.310
Fold 2: Recall=0.387, Precision=0.260, F1=0.311
Fold 3: Recall=0.391, Precision=0.258, F1=0.311
Fold 4: Recall=0.400, Precision=0.257, F1=0.313
Fold 5: Recall=0.399, Precision=0.265, F1=0.319

--- Resumen Validación Cruzada (5 folds, estratificada por target) ---
Recall por fold:    [0.388, 0.387, 0.391, 0.4, 0.399]
Precision por fold: [0.257, 0.26, 0.258, 0.257, 0.265]
F1 por fold:        [0.31, 0.311, 0.311, 0.313, 0.319]

Recall promedio:    0.393 (std: 0.005)
Precision promedio: 0.260 (std: 0.003)
F1 promedio:        0.313 (std: 0.003)


## Guardado del modelo y resumen de resultados

**Corrección 4:** Se guarda en carpeta `models/` dentro del repositorio, no en `../` que saca el archivo del proyecto. Se incluye un archivo JSON con metadatos de calibración para trazabilidad.

In [ ]:
os.makedirs('models', exist_ok=True)

# Guardar modelo y preprocesador
joblib.dump(modelo_corregido, 'models/modelo_logismart_rf.joblib')
joblib.dump(preprocesador, 'models/preprocesador_logismart.joblib')

# Guardar metadatos de calibración
calibration_metadata = {
    'umbral_optimo': float(optimal_threshold),
    'criterio_umbral': 'max_F1 en curva Precision-Recall',
    'recall_global_test': float(recall_score(y_test, y_pred_corregido)),
    'precision_global_test': float(precision_score(y_test, y_pred_corregido)),
    'f1_global_test': float(f1_score(y_test, y_pred_corregido)),
    'recall_nucleo': float(recall_nucleo_v2),
    'recall_periferia': float(recall_periferia_v2),
    'cv_recall_mean': float(np.mean(recalls_fold)),
    'cv_recall_std': float(np.std(recalls_fold)),
    'cv_f1_mean': float(np.mean(f1s_fold)),
    'cv_f1_std': float(np.std(f1s_fold)),
    'nota': 'Umbral calibrado sobre dataset sintético. Recalibrar con datos reales de LogiSmart antes de despliegue.'
}

with open('models/calibracion_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(calibration_metadata, f, indent=2, ensure_ascii=False)

print("Archivos guardados en models/:")
print("  - modelo_logismart_rf.joblib")
print("  - preprocesador_logismart.joblib")
print("  - calibracion_metadata.json")

# Tabla resumen final
resumen_final = pd.DataFrame({
    'metrica': ['Recall global', 'Recall núcleo', 'Recall periferia',
                'Precision global', 'F1 global', 'Umbral óptimo'],
    'baseline': [recall_score(y_test, y_pred_baseline), recall_nucleo, recall_periferia,
                 precision_score(y_test, y_pred_baseline), f1_score(y_test, y_pred_baseline), 0.5],
    'corregido': [recall_score(y_test, y_pred_corregido), recall_nucleo_v2, recall_periferia_v2,
                  precision_score(y_test, y_pred_corregido), f1_score(y_test, y_pred_corregido),
                  round(optimal_threshold, 3)],
})
resumen_final

Archivos guardados en models/:
  - modelo_logismart_rf.joblib
  - preprocesador_logismart.joblib
  - calibracion_metadata.json


,metrica,baseline,corregido
0,Recall global,0.158030,0.394708
1,Recall núcleo,0.087450,0.269693
2,Recall periferia,0.492616,0.987342
3,Precision global,0.339787,0.261346
4,F1 global,0.215728,0.314472
5,Umbral óptimo,0.500000,0.330000


## Conclusiones (insumo para 1.4.3 — Explicabilidad, y para el Informe EP5)

1. El baseline (Random Forest, class_weight balanceado) mostró bajo Recall global con umbral por defecto (0.5), confirmando la necesidad de calibrar el umbral cuando la probabilidad máxima del dataset es inferior a 0.5.
2. La acción correctiva (SMOTENC acotado a periferia + ajuste de umbral vía curva P-R) mejoró el Recall en el segmento periférico y global. Documentar el valor exacto obtenido por el equipo, junto con el trade-off en precisión si lo hubo.
3. La validación cruzada estratificada por target confirma la estabilidad del modelo corregido a través de los 5 folds.
4. El modelo, preprocesador y metadatos de calibración quedan guardados en `models/` como insumo directo para `03_explicabilidad.ipynb` (tarea 1.4.3) y para el dashboard de operaciones (E4).
5. **Limitación declarada:** Las métricas corresponden a un dataset sintético. Antes del despliegue en producción, se debe recalibrar el umbral con datos reales de LogiSmart.